# 01 · Linear algebra for machine learning

**Goal:** get fluent with the four linear-algebra ideas that everything else in this folder leans on, and verify each one numerically.

1. A matrix is a linear map; $A\mathbf{x}$ is a combination of the columns of $A$.
2. Symmetric matrices have an orthogonal eigenbasis: $S = Q\Lambda Q^\top$.
3. The SVD $M = U\Sigma V^\top$ exists for *every* matrix, and truncating it gives the best low-rank approximation (Eckart–Young).
4. PCA is the SVD of centered data.

**Habit to build:** after every derivation, write an `assert` that checks it on random data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## 1. A matrix as a linear map

For $A \in \mathbb{R}^{m\times n}$ and $\mathbf{x}\in\mathbb{R}^n$,

$$A\mathbf{x} = \sum_{j=1}^n x_j\, \mathbf{a}_j ,$$

where $\mathbf{a}_j$ is the $j$-th column of $A$. Matrix–vector multiplication *mixes columns*. Matrix–matrix multiplication $AB$ applies $B$ first and then $A$ (composition of maps).

In [ ]:
A = np.array([[2., 1.], [0., 3.]])
x = np.array([1., 2.])

print("A @ x        =", A @ x)
print("column mix   =", x[0] * A[:, 0] + x[1] * A[:, 1])
assert np.allclose(A @ x, x[0] * A[:, 0] + x[1] * A[:, 1])

# composition: (A B) x = A (B x)
B = rng.normal(size=(2, 2))
assert np.allclose((A @ B) @ x, A @ (B @ x))

## 2. Eigen-decomposition of a symmetric matrix

If $S = S^\top$ then there is an orthogonal $Q$ ($Q^\top Q = I$) and real $\Lambda=\mathrm{diag}(\lambda_i)$ with

$$S = Q \Lambda Q^\top, \qquad S\mathbf{q}_i = \lambda_i \mathbf{q}_i .$$

Matrices of the form $B^\top B$ are symmetric positive semidefinite (all $\lambda_i\ge 0$). These show up constantly: covariance matrices, Hessians of least-squares losses, Gram matrices.

In [ ]:
Bm = rng.normal(size=(4, 4))
S = Bm.T @ Bm                       # symmetric PSD

lam, Q = np.linalg.eigh(S)          # eigh is for symmetric matrices
print("eigenvalues:", lam)

for i in range(4):
    assert np.allclose(S @ Q[:, i], lam[i] * Q[:, i])
assert np.allclose(Q.T @ Q, np.eye(4))                 # orthonormal eigenvectors
assert np.allclose(Q @ np.diag(lam) @ Q.T, S)          # reconstruction
assert np.all(lam >= -1e-12)                           # PSD
print("trace(S) =", np.trace(S).round(4), "  sum(lambda) =", lam.sum().round(4))

## 3. SVD and the best low-rank approximation

Every $M\in\mathbb{R}^{m\times n}$ factors as $M = U\Sigma V^\top$ with orthonormal columns in $U,V$ and $\sigma_1\ge\sigma_2\ge\dots\ge 0$.

**Eckart–Young:** the best rank-$k$ approximation in Frobenius norm is $M_k=\sum_{i\le k}\sigma_i \mathbf{u}_i\mathbf{v}_i^\top$, and

$$\|M - M_k\|_F = \sqrt{\textstyle\sum_{i>k}\sigma_i^2}.$$

Link to eigenvalues: the $\sigma_i^2$ are the eigenvalues of $M^\top M$.

In [ ]:
M = rng.normal(size=(8, 5))
U, s, Vt = np.linalg.svd(M, full_matrices=False)

for k in range(1, 6):
    Mk = U[:, :k] @ np.diag(s[:k]) @ Vt[:k]
    err = np.linalg.norm(M - Mk, "fro")
    pred = np.sqrt(np.sum(s[k:] ** 2))
    print(f"rank {k}: error {err:.4f}   predicted {pred:.4f}")
    assert np.isclose(err, pred)

# sigma_i^2 are eigenvalues of M^T M
assert np.allclose(np.sort(s**2), np.sort(np.linalg.eigvalsh(M.T @ M)))

## 4. PCA is the SVD of centered data

Center the data matrix $X_c\in\mathbb{R}^{n\times d}$. The sample covariance is $C = X_c^\top X_c/(n-1)$. If $X_c = U\Sigma V^\top$ then

$$C = V\,\frac{\Sigma^2}{n-1}\,V^\top,$$

so the columns of $V$ are the principal directions and $\sigma_i^2/(n-1)$ are the variances along them.

In [ ]:
n = 300
Z = rng.normal(size=(n, 2)) @ np.diag([3.0, 0.7])          # axis-aligned cloud
th = np.pi / 6
R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
X = Z @ R.T + np.array([2.0, -1.0])                         # rotate and shift

Xc = X - X.mean(axis=0)
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
var_svd = s**2 / (n - 1)

C = Xc.T @ Xc / (n - 1)
var_eig = np.sort(np.linalg.eigvalsh(C))[::-1]
assert np.allclose(var_svd, var_eig)
print("variance along PCs:", var_svd, " (true: 9.0, 0.49)")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X[:, 0], X[:, 1], s=8, alpha=0.5)
mu = X.mean(axis=0)
for i in range(2):
    v = Vt[i] * 2 * np.sqrt(var_svd[i])
    ax.arrow(mu[0], mu[1], v[0], v[1], color="C3", width=0.05, length_includes_head=True)
ax.set_aspect("equal"); ax.set_title("Principal directions")
plt.show()

## Exercises

1. Show numerically that $\|M\|_2 = \sigma_1$ (the spectral norm is the largest singular value) and that $\|M\|_F^2=\sum\sigma_i^2$.
2. Take a $64\times 64$ matrix with smoothly decaying structure (e.g. $M_{ij}=1/(1+|i-j|)$). Plot $\sigma_i$ and find the smallest $k$ with relative error $<1\%$.
3. Prove (on paper) that $x^\top S x\ge \lambda_{\min}\|x\|^2$ for symmetric $S$, then check it on 1000 random $x$. This inequality is why the condition number controls gradient-descent speed in notebook 02.